# Lesson 10.4: What Is MCP?

*Module 10 · MCP, Tool Orchestration & Function Calling LIVE CLASS*

> Every LLM provider has its own function calling format. MCP — the Model Context Protocol — is the open standard connecting ANY AI model to ANY tool through a single protocol. Created by Anthropic, adopted industry-wide. This lesson covers the spec, architecture, transports, and builds a working MCP server and client.

`MCP Spec` · `JSON-RPC` · `3 Transports` · `Server + Client` · `60 min`

---

## About this notebook

This notebook contains the **2 runnable code examples** from the published lesson `lesson-10.4.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — MCP Messages — JSON-RPC Under the Hood — `01_mcp_messages.py`
2. Step 2 — Three Transports — `02_mcp_server.py`


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · MCP Messages — JSON-RPC Under the Hood

Every MCP interaction is a JSON-RPC 2.0 message.

**`01_mcp_messages.py`** · _Block 1 of 2_

MCP MESSAGE FORMAT — JSON-RPC 2.0


In [ ]:
# MCP MESSAGE FORMAT — JSON-RPC 2.0
import json

# 1. Initialize: client discovers server
init = {"jsonrpc":"2.0", "id":1, "method":"initialize",
        "params": {"protocolVersion":"2025-03-26",
                   "clientInfo":{"name":"netsetos-app"}}}

# 2. List tools: what can this server do?
list_tools = {"jsonrpc":"2.0", "id":2, "method":"tools/list"}

# 3. Call tool: execute a function
call = {"jsonrpc":"2.0", "id":3, "method":"tools/call",
        "params": {"name":"get_weather", "arguments":{"city":"Hyderabad"}}}

# Response
result = {"jsonrpc":"2.0", "id":3,
          "result":{"content":[{"type":"text","text":"34°C, Sunny"}]}}

print("MCP Flow: initialize → tools/list → tools/call → result")
print("Protocol: JSON-RPC 2.0 over any transport")


> **What just happened?** Three JSON-RPC messages = the core MCP flow: initialize (handshake), tools/list (discover), tools/call (execute). This is the ENTIRE protocol. Any transport carries these same messages.


### Step 2 · Three Transports

Same protocol, three delivery mechanisms.

**`02_mcp_server.py`** · _Block 2 of 2_

MCP SERVER — Build a working server — pip install mcp


In [ ]:
# MCP SERVER — Build a working server
# pip install mcp
from mcp.server import Server
from mcp.server.stdio import stdio_server
from mcp.types import Tool, TextContent

app = Server("netsetos-tools")

@app.list_tools()
async def list_tools():
    return [
        Tool(name="get_weather", description="Get weather for a city",
             inputSchema={"type":"object","properties":{"city":{"type":"string"}},"required":["city"]}),
        Tool(name="search_courses", description="Search Netsetos courses",
             inputSchema={"type":"object","properties":{"topic":{"type":"string"}},"required":["topic"]}),
    ]

@app.call_tool()
async def call_tool(name, arguments):
    if name == "get_weather":
        return [TextContent("text", f"34°C Sunny in {arguments['city']}")]
    elif name == "search_courses":
        return [TextContent("text", "GenAI Complete: 14999 INR")]

print("MCP Server: @app.list_tools + @app.call_tool")
print("Run via stdio, add to claude_desktop_config.json")


> **What just happened?** The three-layer architecture (host/client/server) is MCP's security foundation. The host mediates all access and enforces policies. Each client maintains exactly one session with one server — no cross-server communication. The LLM interacts with tools through the client bridge , which translates MCP tools into the model's native function calling format.


---

## Wrap-up

You've walked through all 2 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
